### DATA CLEANING & FEATURE ENGINEERING

## 0. Setup


In [16]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [18]:
# Load raw data
df_raw = pd.read_excel("/content/gdrive/MyDrive/STAT3013/Dataset/Extracted-Data.xlsx")
print(f"Raw shape: {df_raw.shape}")
print(f"Columns: {df_raw.shape[1]}")
df_raw["outcome"].value_counts()

Raw shape: (710, 95)
Columns: 95


,count
outcome,
Strength,490
Hypertrophy,220


## 1. Filter - Chỉ giữ Hypertrophy

In [19]:
# Filter outcome == Hypertrophy TRƯỚC
df = df_raw[df_raw["outcome"] == "Hypertrophy"].copy()
df = df.reset_index(drop=True)

print(f"After filter: {df.shape[0]} rows (từ {df_raw.shape[0]} rows)")
print(f"Outcome values: {df['outcome'].unique()}")

After filter: 220 rows (từ 710 rows)
Outcome values: ['Hypertrophy']


## 2. Chọn cột (29 cột từ 95 cột)

In [20]:
STUDY_INFO = [
    "number", "first.author", "year", "design", "upper.lower"
]

SUBJECT = [
    "age", "sex.male", "train.status", "body.mass", "years.exp"
]

TRAINING = [
    "sets.week.all", "sets.week.direct",
    "frequency.direct", "sessions.per.week",
    "reps.week.all", "rep.range.all",
    "interset.rest.min.all", "percentage.failure.all",
    "failure.nonfailure.mixed.direct", "weeks"
]

NUTRITION = [
    "nutrition.controlled.surplus?", "nutrition.controlled.deficit?"
]

# Cần để tính Hedges g (không đưa vào ML)
COMPUTE_G = [
    "pre.mean", "post.mean", "pre.sd", "post.sd", "n",
    "measure", "outcome"
]

ALL_COLS = STUDY_INFO + SUBJECT + TRAINING + NUTRITION + COMPUTE_G
df = df[ALL_COLS].copy()

print(f"Selected: {len(ALL_COLS)} columns")
print(f"Shape: {df.shape}")

Selected: 29 columns
Shape: (220, 29)


## 3. Kiểm tra missing values

In [21]:
# Missing % cho từng cột
miss = (df.isnull().sum() / len(df) * 100).round(1).sort_values(ascending=False)
miss_df = pd.DataFrame({"Column": miss.index, "Missing %": miss.values})
print(miss_df[miss_df["Missing %"] > 0].to_string(index=False))

                         Column  Missing %
                      years.exp       45.0
                         pre.sd       10.0
                        post.sd       10.0
                      body.mass        4.5
         percentage.failure.all        4.5
          interset.rest.min.all        3.6
                       sex.male        2.7
failure.nonfailure.mixed.direct        1.8


## 4. Xử lí missing values

In [22]:
# Các cột numeric có thể chứa string → ép về số trước
NUMERIC_COLS = [
    "years.exp", "body.mass", "sex.male",
    "rep.range.all", "interset.rest.min.all", "percentage.failure.all"
]
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ── years.exp: 45% missing → impute median ────────────────────
median_exp = df["years.exp"].median()
df["years.exp"] = df["years.exp"].fillna(median_exp)
print(f"years.exp imputed with median = {median_exp:.2f}")

# ── body.mass: 4.5% missing → impute median ──────────────────
median_bm = df["body.mass"].median()
df["body.mass"] = df["body.mass"].fillna(median_bm)
print(f"body.mass imputed with median = {median_bm:.2f}")

# ── sex.male: 2.7% missing → impute median ───────────────────
median_sex = df["sex.male"].median()
df["sex.male"] = df["sex.male"].fillna(median_sex)
print(f"sex.male imputed with median = {median_sex:.2f}")
# FIX: sex.male trong dataset gốc là % (0-100) → normalize về 0-1
# Lý do: các ML model cần features cùng scale; 100 >> 1 gây bias
df["sex.male"] = df["sex.male"] / 100.0
print(f"sex.male normalized → range: [{df['sex.male'].min():.2f}, {df['sex.male'].max():.2f}]")

# ── rep.range.all: 4.5% missing → impute median ──────────────
df["rep.range.all"] = df["rep.range.all"].fillna(df["rep.range.all"].median())

# ── interset.rest.min.all: 3.6% → impute median ──────────────
df["interset.rest.min.all"] = df["interset.rest.min.all"].fillna(df["interset.rest.min.all"].median())

# ── percentage.failure.all: 4.5% → impute 0 (no failure data = 0%) ─
df["percentage.failure.all"] = df["percentage.failure.all"].fillna(0)

# ── failure.nonfailure.mixed.direct: 1.8% → impute mode ─────
mode_fail = df["failure.nonfailure.mixed.direct"].mode()[0]
df["failure.nonfailure.mixed.direct"] = df["failure.nonfailure.mixed.direct"].fillna(mode_fail)

print(f"Missing sau impute:")
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else "Không còn missing!")

years.exp imputed with median = 3.50
body.mass imputed with median = 77.50
sex.male imputed with median = 100.00
sex.male normalized → range: [0.27, 1.00]
Missing sau impute:
pre.sd     22
post.sd    22
dtype: int64


## 5. Tính Hedges'g (Target variable)

In [23]:
def compute_hedges_g(row):
    """Tính Hedges g từ pre/post means và SDs"""
    try:
        pre_m  = row["pre.mean"]
        post_m = row["post.mean"]
        pre_sd = row["pre.sd"]
        post_sd= row["post.sd"]
        n      = row["n"]

        # Cần đủ 5 giá trị
        if any(pd.isna(v) for v in [pre_m, post_m, pre_sd, post_sd, n]):
            return np.nan

        # Pooled SD
        pooled = np.sqrt((pre_sd**2 + post_sd**2) / 2)
        if pooled == 0:
            return np.nan

        # Cohen d
        d = (post_m - pre_m) / pooled

        # Hedges correction factor J
        J = 1 - 3 / (4 * (2 * n - 2) - 1)

        return round(d * J, 4)
    except:
        return np.nan

df["hedges_g"] = df.apply(compute_hedges_g, axis=1)

# Drop rows không tính được g
n_before = len(df)
df = df.dropna(subset=["hedges_g"]).reset_index(drop=True)
n_after = len(df)

print(f"Hedges g computed: {n_after} / {n_before} rows")
print(f"Hedges g stats:")
print(df["hedges_g"].describe().round(4))

Hedges g computed: 198 / 220 rows
Hedges g stats:
count    198.0000
mean       0.4291
std        0.3698
min       -0.2080
25%        0.1454
50%        0.3602
75%        0.6146
max        2.2588
Name: hedges_g, dtype: float64


## 6. Loại outliers(|g| > 3)

In [24]:
n_before = len(df)
df = df[df["hedges_g"].abs() <= 3].reset_index(drop=True)
n_after = len(df)
print(f"Outliers removed: {n_before - n_after} rows")
print(f"Final shape: {df.shape}")
print(f"hedges_g range: [{df['hedges_g'].min():.4f}, {df['hedges_g'].max():.4f}]")

Outliers removed: 0 rows
Final shape: (198, 30)
hedges_g range: [-0.2080, 2.2588]


## 7. Feature Engineering

In [25]:
# 7.1 volume_category
def vol_cat(x):
    if x <= 6:   return "Low"
    elif x <= 15: return "Moderate"
    else:         return "High"

df["volume_category"] = df["sets.week.all"].apply(vol_cat)
df["volume_category"] = pd.Categorical(df["volume_category"],
                         categories=["Low","Moderate","High"], ordered=True)
print("volume_category:")
print(df["volume_category"].value_counts().sort_index())

volume_category:
volume_category
Low         48
Moderate    61
High        89
Name: count, dtype: int64


In [26]:
# 7.2 failure_binary
def fail_bin(x):
    if str(x).lower() == "failure": return 1
    return 0

df["failure_binary"] = df["failure.nonfailure.mixed.direct"].apply(fail_bin)
print("failure_binary:")
print(df["failure_binary"].value_counts())

failure_binary:
failure_binary
1    159
0     39
Name: count, dtype: int64


In [27]:
# 7.3 nutrition_category
def nutr_cat(row):
    s = str(row["nutrition.controlled.surplus?"]).strip().lower()
    d = str(row["nutrition.controlled.deficit?"]).strip().lower()
    if s == "yes": return "Surplus"
    if d == "yes": return "Deficit"
    return "None"

df["nutrition_category"] = df.apply(nutr_cat, axis=1)
print("nutrition_category:")
print(df["nutrition_category"].value_counts())

# FIX 2: binary flag thay nutrition_category cho ML
df["has_nutrition_control"] = (
    df["nutrition_category"].isin(["Surplus", "Deficit"])
).astype(int)
print(f"\n[FIX] has_nutrition_control:")
print(df["has_nutrition_control"].value_counts())

nutrition_category:
nutrition_category
None       188
Surplus      6
Deficit      4
Name: count, dtype: int64

[FIX] has_nutrition_control:
has_nutrition_control
0    188
1     10
Name: count, dtype: int64


## 8. Chuẩn hóa dtype

In [28]:
# Encode train.status → số cho ML (giữ cột gốc)
train_map = {"untrained": 0, "recreational": 1, "trained": 2}
df["train_status_enc"] = df["train.status"].str.lower().map(train_map).fillna(1).astype(int)

# upper.lower → binary
df["upper_body"] = (df["upper.lower"].str.lower() == "upper").astype(int)

# hyp_class — phân loại mức hypertrophy
def hyp_class(g):
    if g < 0.2:  return "Low"
    elif g < 0.8: return "Medium"
    else:         return "High"

df["hyp_class"] = df["hedges_g"].apply(hyp_class)
df["hyp_class"] = pd.Categorical(df["hyp_class"],
                   categories=["Low","Medium","High"], ordered=True)

print("hyp_class (ML classification target):")
print(df["hyp_class"].value_counts().sort_index())
print(f"train_status_enc:")
print(df["train_status_enc"].value_counts())

# FIX 3: làm tròn tất cả float columns về 2 chữ số thập phân
INT_COLS = ["number","year","n","failure_binary",
            "has_nutrition_control","train_status_enc","upper_body"]
for col in df.columns:
    if col not in INT_COLS and df[col].dtype in ["float64","float32"]:
        df[col] = df[col].round(2)
print("[FIX] Float columns làm tròn 2 chữ số thập phân")

hyp_class (ML classification target):
hyp_class
Low        60
Medium    112
High       26
Name: count, dtype: int64
train_status_enc:
train_status_enc
2    117
0     81
Name: count, dtype: int64
[FIX] Float columns làm tròn 2 chữ số thập phân


## 9. Lưu file output

In [29]:
# FIX 5: bỏ nutrition_category (188/198 missing) khi save
df_save = df.drop(columns=["nutrition_category"], errors="ignore")
df_save.to_csv("data_cleaned.csv", index=False)
print(f"Saved data_cleaned.csv: {df_save.shape}")

# data_features.csv — chỉ features + target cho ML
# FIX 4: bỏ nutrition_category + volume_category, thêm has_nutrition_control
FEATURES = [
    "sets.week.all", "sets.week.direct",
    "frequency.direct", "sessions.per.week",
    "reps.week.all", "rep.range.all",
    "interset.rest.min.all", "percentage.failure.all",
    "failure_binary", "weeks",
    "train_status_enc", "sex.male", "age", "upper_body",
    "has_nutrition_control",
    "hedges_g", "hyp_class"
]
df_feat = df[FEATURES].copy()
df_feat.to_csv("data_features.csv", index=False)
print(f"Saved data_features.csv: {df_feat.shape}")

# Summary
print(f"{'='*50}")
print(f"FINAL SUMMARY:")
print(f"  Raw rows:        710")
print(f"  After filter:    220 (Hypertrophy only)")
print(f"  After hedges_g:  {df.shape[0]} rows usable")
print(f"  Features:        {len(FEATURES)-2} input + 2 targets")
print(f"  Volume Low:      {(df['volume_category']=='Low').sum()}")
print(f"  Volume Moderate: {(df['volume_category']=='Moderate').sum()}")
print(f"  Volume High:     {(df['volume_category']=='High').sum()}")

Saved data_cleaned.csv: (198, 36)
Saved data_features.csv: (198, 17)
FINAL SUMMARY:
  Raw rows:        710
  After filter:    220 (Hypertrophy only)
  After hedges_g:  198 rows usable
  Features:        15 input + 2 targets
  Volume Low:      48
  Volume Moderate: 61
  Volume High:     89
